# 🚀 OpenHands AI Agent trên Google Colab

Notebook này cho phép bạn chạy **OpenHands AI Agent** trực tiếp trên Google Colab kết hợp với mô hình **Gemini 3 Flash**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tuna1710/openhands-workspace/blob/main/openhands_colab.ipynb)

> **Lưu ý kỹ thuật:** Máy ảo Google Colab chạy trong môi trường container giới hạn không có sẵn Docker daemon. Do đó, notebook này sử dụng **OpenHands Native Process & CLI Mode** (`RUNTIME=process`), cho phép Agent thao tác trực tiếp với hệ điều hành và file của Colab một cách mượt mà nhất.

## 📦 Bước 1: Cài đặt OpenHands & Cloudflared

In [ ]:
# 1. Cài đặt OpenHands CLI & Agent SDK
!pip install -q openhands-ai nest_asyncio

# 2. Tải Cloudflared để mở cổng Web UI ra Internet (hoàn toàn miễn phí, không cần đăng ký tài khoản)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ Cài đặt thành công!")

## 🔑 Bước 2: Nhập API Key & Thiết lập Model Gemini

In [ ]:
import os
from getpass import getpass

# Nhập Gemini API Key từ Google AI Studio (https://aistudio.google.com/)
api_key = getpass("🔑 Nhập Google Gemini API Key của bạn: ")
os.environ["GEMINI_API_KEY"] = api_key
os.environ["LLM_API_KEY"] = api_key
os.environ["LLM_MODEL"] = "gemini/gemini-3.8-flash"
os.environ["RUNTIME"] = "process"  # Chế độ chạy trực tiếp không cần Docker

print("✅ Đã cấu hình Model: gemini/gemini-3.8-flash")

## 🎯 Cách 1: Giao việc trực tiếp cho Agent ngay trong Colab (Nhanh & Ổn định nhất)

In [ ]:
# Đổi nội dung task theo ý muốn của bạn
task = "Hãy tạo một ứng dụng web Game rắn săn mồi (Snake Game) với HTML, CSS và JavaScript, sau đó viết file README.md giải thích code."

!openhands --prompt "{task}" --model "gemini/gemini-3.8-flash"

## 🌐 Cách 2: Khởi chạy Giao diện Web (OpenHands Web UI) với Cloudflare Tunnel
Chạy ô bên dưới để lấy link mở giao diện web đầy đủ của OpenHands:

In [ ]:
import subprocess
import time
import re

# 1. Khởi chạy OpenHands Server ngầm trên cổng 3000
server_process = subprocess.Popen(
    ["openhands", "serve", "--port", "3000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
print("⏳ Đang khởi động OpenHands server...")
time.sleep(10)

# 2. Mở đường hầm Cloudflare Tunnel công khai
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:3000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("🔗 Đang tạo link truy cập Web UI...")
for line in tunnel_process.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        print(f"\n🎉 BẤM VÀO ĐÂY ĐỂ MỞ OPENHANDS WEB UI:\n👉 {public_url}\n")
        break